# Chapitre 2 — Construire un premier RAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-02-premier-rag/02_premier_rag.ipynb)

Ce notebook met en œuvre ingestion, chunking, embeddings OpenAI, retrieval, génération OpenAI et citations.

## Scripts du chapitre

1. [`01_document_loading.py`](examples/01_document_loading.py)
2. [`02_pipeline_ingestion.py`](examples/02_pipeline_ingestion.py)
3. [`03_generator_rag.py`](examples/03_generator_rag.py)
4. [`04_naive_rag_complet.py`](examples/04_naive_rag_complet.py)

## 1. Préparer le dépôt

La cellule fonctionne dans Colab et depuis la racine du dépôt.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

sys.path.insert(0, str(Path("src").resolve()))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Dépôt prêt :", Path.cwd())


## 2. Configurer OpenAI

La clé est saisie de manière masquée et n'est jamais enregistrée dans le notebook.

In [ ]:
from getpass import getpass
import os

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
if not os.getenv("OPENAI_MODEL"):
    os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()

if not os.environ["OPENAI_API_KEY"] or not os.environ["OPENAI_MODEL"]:
    raise RuntimeError("OPENAI_API_KEY et OPENAI_MODEL sont obligatoires")
print("Configuration OpenAI chargée.")


## 3. Charger les documents

Correspond à [`01_document_loading.py`](examples/01_document_loading.py).

In [ ]:
from pathlib import Path
from rag_en_pratique.core import Document

data_directory = Path("data/sample")
documents = [
    Document(path.read_text(encoding="utf-8"), {"source": path.name})
    for path in sorted(data_directory.glob("*.md"))
    if path.name.lower() != "readme.md"
]
[(doc.metadata["source"], len(doc.text)) for doc in documents]


## 4. Découper les documents

In [ ]:
from rag_en_pratique.core import split_documents

chunks = split_documents(documents, chunk_size=60, overlap=10)
print(f"{len(documents)} documents -> {len(chunks)} chunks")
chunks[0]


## 5. Indexer avec OpenAI et rechercher

Correspond à [`02_pipeline_ingestion.py`](examples/02_pipeline_ingestion.py).

In [ ]:
from rag_en_pratique.core import InMemoryVectorStore
from rag_en_pratique.openai_adapter import OpenAIEmbedder

store = InMemoryVectorStore(OpenAIEmbedder())
store.add(chunks)
results = store.search("Quel est le délai pour retourner un produit ?", top_k=3)
[(round(item.score, 3), item.document.metadata["source"]) for item in results]


## 6. Générer la réponse avec OpenAI

Correspond à [`03_generator_rag.py`](examples/03_generator_rag.py) et [`04_naive_rag_complet.py`](examples/04_naive_rag_complet.py).

In [ ]:
from rag_en_pratique.core import RAGPipeline
from rag_en_pratique.openai_adapter import OpenAIGenerator

rag = RAGPipeline(store, OpenAIGenerator())
response = rag.ask("Sous combien de jours peut-on retourner un produit ?")
print(response["answer"])


## 7. Examiner les sources

Une application RAG doit rendre ses sources inspectables.

In [ ]:
for source in response["sources"]:
    print(source["score"], source["metadata"]["source"])
    print(source["text"][:250])
    print()


## Pour aller plus loin

Comparez plusieurs tailles de chunks, modifiez `top_k`, ajoutez un document et vérifiez que la réponse cite toujours la bonne source.